# TFI - GRUPO 3



## Paso 0 — Dónde estás parado



In [2]:
from google.cloud import storage

# Proyecto de GCP 
PROJECT_ID = "mma-cloudproject"

REGION = "us-central1"
BUCKET = f"{PROJECT_ID}-tfi-grupo3"

# El cliente autentica solo con las credenciales de Cloud Shell (ADC).
gcs = storage.Client(project=PROJECT_ID)

print("Proyecto:", PROJECT_ID)
print("Region:  ", REGION)
print("Bucket:  ", BUCKET)

Proyecto: mma-cloudproject
Region:   us-central1
Bucket:   mma-cloudproject-tfi-grupo3


## Paso 1 — La nube te responde

Un proyecto de GCP nace con casi todo **apagado**: cada servicio hay que **habilitar su API**. Las de
hoy (Vertex, Cloud Storage, Artifact Registry) las prendimos en la **Parte 0** desde la consola.

Para comprobar que todo está en orden, le pedimos a Cloud Storage la lista de buckets del proyecto.
Si esto responde, la API está prendida y estás autenticado.

> Si te faltara alguna API, se prende desde la consola (*APIs y servicios → Habilitar*) o en la
> terminal con `gcloud services enable aiplatform.googleapis.com storage.googleapis.com`.

In [3]:
# Si esto lista buckets (aunque sea vacío), Cloud Storage responde y estas autenticado.
print("Buckets del proyecto:")
for b in gcs.list_buckets():
    print(" -", b.name)

Buckets del proyecto:
 - mma-cloudproject-churn
 - mma-cloudproject-tfi-grupo3
 - mma-cloudproject_cloudbuild


## Paso 2 — El dato a la nube

El dato deja de vivir en una computadora y pasa a **Cloud Storage**, el almacenamiento de objetos de GCP
(el equivalente al S3 de AWS). Primero verificamos que el CSV canónico esté sano (forma y checksum),
después creamos un **bucket** y subimos el archivo.

**Andá a ver:** en la consola, *Cloud Storage → Buckets → tu bucket → `raw/`*. Ahí está tu CSV.

In [7]:
import sys, subprocess     

# Creamos el bucket si no existe.
bucket = gcs.bucket(BUCKET)
if bucket.exists():
    print("El bucket ya existe:", BUCKET)
else:
    bucket = gcs.create_bucket(BUCKET, location=REGION)
    print("Bucket creado:", BUCKET)


El bucket ya existe: mma-cloudproject-tfi-grupo3


## Paso 3 — Entrenar

Acá pasás de **datos** a **modelo**. Reutilizamos el pipeline del caso guía (imputación, escalado,
one-hot y una regresión logística balanceada). Entrena en **segundos**: es el camino *a mano*, rápido
y reproducible. Vertex AutoML haría algo parecido, pero tardaría horas; lo miramos al final.

Cuando termine, leé el **AUC** y el **recall**: el recall es qué proporción de los que se van de
verdad estás agarrando. Para CRM, un recall alto significa **no dejar pasar** clientes en riesgo.

In [12]:
import sys

print("Python del notebook:")
print(sys.executable)

import torch
import ultralytics

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Ultralytics:", ultralytics.__version__)

Python del notebook:
/bin/python
PyTorch: 2.14.0+cpu
CUDA: False
Ultralytics: 8.4.87


In [13]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

print("Modelo cargado correctamente")

Modelo cargado correctamente


In [14]:
from pathlib import Path
import shutil

import yaml
import pandas as pd

PROJECT_ROOT = Path.cwd()

BUCKET_NAME = "mma-cloudproject-tfi-grupo3"
BUCKET_DATA_PREFIX = f"gs://{BUCKET_NAME}/processed/deep_pcb_yolo/"

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "deep_pcb_yolo"
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Sincroniza el dataset procesado del bucket a disco local.
!gcloud storage rsync -r "{BUCKET_DATA_PREFIX}" "{DATA_PROCESSED_DIR}"

# El archivo de configuración puede llamarse data.yaml o data.yml según cómo se haya
# generado; probamos ambos nombres.
DATA_YAML_PATH = DATA_PROCESSED_DIR / "data.yaml"
if not DATA_YAML_PATH.exists():
    DATA_YAML_PATH = DATA_PROCESSED_DIR / "data.yml"

assert DATA_YAML_PATH.exists(), f"No se encontró data.yaml/data.yml en {DATA_PROCESSED_DIR}"
assert (DATA_PROCESSED_DIR / "images").exists(), "No se encontró la carpeta images/"
assert (DATA_PROCESSED_DIR / "labels").exists(), "No se encontró la carpeta labels/"

# Nos aseguramos de que el data.yaml apunte a la ruta local donde acabamos de
# descargar el dataset (por si el archivo fue generado en otra máquina/entorno).
with open(DATA_YAML_PATH, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

data_yaml["path"] = str(DATA_PROCESSED_DIR)

with open(DATA_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("Dataset sincronizado en:", DATA_PROCESSED_DIR)
print("data.yaml:", DATA_YAML_PATH)
print(data_yaml)

At gs://mma-cloudproject-tfi-grupo3/processed/deep_pcb_yolo/**, worker process 4261 thread 138369346549568 listed 3003...
Copying gs://mma-cloudproject-tfi-grupo3/processed/deep_pcb_yolo/data.yaml to file:///home/mapcabezas/pcb-defect-detection-yolo/notebooks/data/processed/deep_pcb_yolo/data.yaml
Copying gs://mma-cloudproject-tfi-grupo3/processed/deep_pcb_yolo/images/test/00041200.jpg to file:///home/mapcabezas/pcb-defect-detection-yolo/notebooks/data/processed/deep_pcb_yolo/images/test/00041200.jpg
Copying gs://mma-cloudproject-tfi-grupo3/processed/deep_pcb_yolo/images/test/00041201.jpg to file:///home/mapcabezas/pcb-defect-detection-yolo/notebooks/data/processed/deep_pcb_yolo/images/test/00041201.jpg
Copying gs://mma-cloudproject-tfi-grupo3/processed/deep_pcb_yolo/images/test/00041202.jpg to file:///home/mapcabezas/pcb-defect-detection-yolo/notebooks/data/processed/deep_pcb_yolo/images/test/00041202.jpg
Copying gs://mma-cloudproject-tfi-grupo3/processed/deep_pcb_yolo/images/test/000

In [15]:
import time

RUNS_DIR = PROJECT_ROOT / "runs"

MODEL_VARIANT = "yolo11s"
RUN_NAME = f"{MODEL_VARIANT}_deeppcb"
TEST_RUN_NAME = f"{MODEL_VARIANT}_test"

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("data.yaml:", DATA_YAML_PATH)

CUDA disponible: False
data.yaml: /home/mapcabezas/pcb-defect-detection-yolo/notebooks/data/processed/deep_pcb_yolo/data.yaml


In [16]:
start_time = time.time()

model = YOLO(f"{MODEL_VARIANT}.pt")

results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    workers=2,
    seed=42,
    patience=10,
    exist_ok=True
)

elapsed_time = time.time() - start_time

print(f"Tiempo total {MODEL_VARIANT}: {elapsed_time / 60:.2f} minutos")

New https://pypi.org/project/ultralytics/8.4.146 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.87 🚀 Python-3.12.3 torch-2.14.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


## Paso 4 — Registrar y versionar el modelo

Un modelo sin registrar es un archivo perdido. Le ponemos una **versión** con fecha y guardamos el
artefacto más sus métricas. Después lo subimos a Cloud Storage: así el modelo también **vive en la
nube**, con su linaje.

> Nota: el **Model Registry gestionado** de Vertex, para un modelo propio, pide además un contenedor
> de serving. Esa complejidad la dejamos para más adelante. Hoy el "registro" es el artefacto **versionado
> en Cloud Storage**, y el Registry real lo vas a ver en la demo de AutoML.

In [ ]:
import json
import joblib
from datetime import datetime, timezone

version = datetime.now(timezone.utc).strftime("churn-baseline-%Y%m%dT%H%M%SZ")
Path("../models").mkdir(parents=True, exist_ok=True)

joblib.dump(
    {"pipeline": pipeline, "metadata": {"model_version": version}},
    "../models/churn-baseline.joblib",
)
json.dump(
    {"model_version": version, "roc_auc": float(roc_auc_score(y_te, proba))},
    open("../models/churn-baseline-metrics.json", "w"),
    indent=2,
)
print("Modelo versionado:", version)

In [ ]:
# Subimos el modelo versionado y sus metricas a la nube.
bucket = gcs.bucket(BUCKET)
bucket.blob("models/churn-baseline.joblib").upload_from_filename("../models/churn-baseline.joblib")
bucket.blob("models/churn-baseline-metrics.json").upload_from_filename("../models/churn-baseline-metrics.json")

print("Modelo en gs://%s/models/" % BUCKET)
for b in gcs.list_blobs(BUCKET, prefix="models/"):
    print(" -", b.name)

## Paso 5 — Scoring batch

Esta es la inferencia **batch**: en vez de responder de a un cliente, scoreamos a **todos** de una
(acá usamos el conjunto de test como si fueran los clientes activos del mes) y armamos el ranking.

La prioridad no es solo la probabilidad de baja: es `churn_probability × MonthlyCharges`. Así CRM
llama primero a quien **se va y factura mucho**.

**Andá a ver:** después de subir el resultado, aparece en *Cloud Storage → `scored/`*.

In [ ]:
scored = X_te.copy()
scored["customerID"] = df.loc[X_te.index, "customerID"]
scored["churn_probability"] = proba
scored["priority_score"] = scored["churn_probability"] * scored["MonthlyCharges"]

ranking = scored.sort_values("priority_score", ascending=False)

# Guardamos el resultado del batch.
Path("../data/scored").mkdir(parents=True, exist_ok=True)
ranking.to_csv("../data/scored/batch-scored.csv", index=False)

ranking[["customerID", "churn_probability", "MonthlyCharges", "priority_score"]].head(10)

In [ ]:
# El resultado del batch tambien va a la nube.
gcs.bucket(BUCKET).blob("scored/batch-scored.csv").upload_from_filename(
    "../data/scored/batch-scored.csv"
)
print("Ranking en gs://%s/scored/" % BUCKET)
for b in gcs.list_blobs(BUCKET, prefix="scored/"):
    print(" -", b.name)

## Paso 6 — De score a decisión

El número no vale solo: vale porque **ordena una acción**. Aplicamos el umbral del caso: los clientes
con `churn_probability >= 0.7` entran a la **cola de retención**; al resto lo monitoreamos.

Esta es la lista concreta que CRM abre el lunes a la mañana.

In [ ]:
umbral = 0.7
cola = ranking[ranking["churn_probability"] >= umbral]

print(f"Clientes en cola de retencion (P >= {umbral}): {len(cola)} de {len(ranking)}")
cola[["customerID", "churn_probability", "MonthlyCharges", "priority_score"]].head(10)

## Paso 7 — Costos y limpieza

Regla de oro de la nube: **lo que prendés, cuesta**. Hoy tuvimos suerte: no desplegamos ningún
endpoint ni dejamos ninguna máquina prendida, así que no hay costo de serving corriendo. Lo único que
queda es el bucket con unos pocos MB, que entra holgado en el free tier.

La celda de abajo **borra los artefactos locales** que generó el lab (el modelo, sus métricas y el CSV
scoreado): ya no hacen falta en disco porque **el modelo vive en la nube** (`gs://…/models/`), que es
la copia que vale y desde donde queda disponible para la **clase 5**. El **bucket se conserva** (por eso
su borrado queda comentado): borrarlo se lo dejamos a quien no vaya a seguir con la materia.

In [ ]:
from pathlib import Path

# Recursos en la NUBE: el bucket se CONSERVA. El modelo vive ahi (gs://.../models/) y
# desde ahi queda disponible para la clase 5. Descomenta solo si NO vas a seguir:
# gcs.bucket(BUCKET).delete(force=True)

# Artefactos LOCALES que genero el lab: los borramos, ya no hacen falta en disco.
for p in [Path("../models/churn-baseline.joblib"),
          Path("../models/churn-baseline-metrics.json"),
          Path("../data/scored/batch-scored.csv")]:
    p.unlink(missing_ok=True)
    print("borrado local:", p)

print("Listo. Nada quedo prendido cobrando; el modelo persiste en gs://%s/models/." % BUCKET)

## Bonus — AutoML, para que lo veas (concepto)

Lo que hicimos a mano en segundos, Vertex lo puede hacer **solo**: le das el dataset tabular, le
decís que la columna a predecir es `Churn`, y prueba modelos por vos. El resultado es parecido; el
costo es **tiempo y crédito** (el entrenamiento tarda alrededor de dos horas), por eso no lo corremos
en vivo. En el ensayo, AutoML sacó **ROC AUC 0.895**, un poco mejor que nuestro `0.842` a mano.

El flujo gestionado, para que lo ubiques, es este (en clase lo vemos con capturas ya preparadas):

```text
1) Crear un dataset tabular en Vertex desde el CSV en Cloud Storage.
2) Entrenar una clasificacion binaria con target = Churn.
3) Revisar las metricas de evaluacion que Vertex calcula solo.
4) Registrar el modelo en el Model Registry.
5) Correr un batch prediction sobre los clientes activos.
```

### ¿Querés reproducirlo vos?

No desde esta notebook: el AutoML se lanza **desde la TERMINAL** de Cloud Shell (ahí el SDK
`aiplatform` tiene proyecto y credenciales; el kernel del editor no). Con el dataset tabular ya creado
en la consola, se corre con dos scripts del repo:

```bash
pip install -q "google-cloud-aiplatform>=1.70,<2"

# Entrena AutoML (async, ~2 h server-side). Pasale el ID de tu dataset tabular:
DATASET_ID=<id-del-dataset> python ../scripts/train_automl.py

# Cuando el modelo 'churn-automl' aparezca en Registro de modelos, scorea a todos:
MODEL_ID=<id-del-modelo> python ../scripts/batch_predict_automl.py
```

> **Ojo:** usá estos scripts (SDK), no la opción "AutoML en canalizaciones" de la consola: ese
> template de Google falla por un bug propio. El detalle y los comandos parametrizados están en
> `../gcp/runbook.md`, sección *Clase 4*.

## Cierre

Recorriste el ciclo entero de una punta: **dato → modelo → registro → batch → decisión**. De un CSV
salió una lista priorizada para CRM.

En la **clase 5** damos vuelta la pregunta: ¿cómo hace *otro sistema* para pedirle el score a este
modelo en el momento, sin abrir esta notebook? Ese mismo modelo, detrás de una **API** con contrato
estable.

**Tarea:** corré esta misma notebook con el **dato de tu TFI**. Con que entrenes algo y veas un
score, alcanza. Anotá qué te rompió.